# 05 - Model Evaluation

Evaluate the **production** model (`models/fraud_model.pkl`, written by
`python main.py`) on the exact rows it never saw during training:

1. stored metrics (`models/evaluation_results.json`)
2. live re-evaluation + confusion matrix
3. ROC and Precision-Recall curves
4. the rule engine on the same rows
5. the combined risk score from a business (alert-volume) perspective

**Prerequisite:** `python main.py` (produces `data/processed/` + `models/`).
Some cells stream the held-out rows out of the 499 MB cleaned file, so they
take ~30-60 s on first run.

In [ ]:
import sys
from pathlib import Path

# Make 'import src...' work no matter where the kernel was started from.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_transactions.csv"
FEATURED_PATH = PROJECT_ROOT / "data" / "processed" / "featured_transactions.csv"
MODEL_PATH = PROJECT_ROOT / "models" / "fraud_model.pkl"

print("Project root:", PROJECT_ROOT)

## 1. Stored metrics

In [ ]:
import json

EVAL_PATH = PROJECT_ROOT / "models" / "evaluation_results.json"
results = json.loads(EVAL_PATH.read_text()) if EVAL_PATH.exists() else None

if results:
    scalar = {k: v for k, v in results.items()
              if k not in ("confusion_matrix", "classification_report")}
    pd.Series(scalar).to_frame("last recorded run")
else:
    print("No stored results yet - run the evaluation in section 2 (or 'python main.py').")

## 2. Live re-evaluation on the held-out rows

`evaluate_on_test_set()` loads **only** the test positions recorded in the
model bundle (streamed from disk, never the whole file in RAM), re-engineers
their features with the persisted threshold, scores them and rewrites
`models/evaluation_results.json`.

In [ ]:
from src.evaluation.model_evaluation import evaluate_on_test_set

results = evaluate_on_test_set()
print(results["classification_report"])

## 3. Confusion matrix

On an imbalanced problem the confusion matrix tells the real story: with a
~0.1% base rate, "99.8% accuracy" is reachable by predicting *legitimate*
every single time. What matters is the recall/precision trade-off on the
fraud class.

In [ ]:
cm = np.array(results["confusion_matrix"])
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm, cmap="Blues")
for (i, j), v in np.ndenumerate(cm):
    ax.text(j, i, f"{int(v):,}", ha="center", va="center",
            color="black" if v < cm.max() * 0.6 else "white")
ax.set_xticks([0, 1], ["legit", "fraud"])
ax.set_yticks([0, 1], ["legit", "fraud"])
ax.set_xlabel("predicted")
ax.set_ylabel("actual")
ax.set_title("Production model - held-out rows")
plt.tight_layout()
plt.show()

print(f"fraud caught (recall):           {tp}/{tp + fn} = {tp / (tp + fn):.3f}")
print(f"alerts that are real (precision): {tp}/{tp + fp} = {tp / max(1, tp + fp):.3f}")
print(f"legit rows wrongly alerted:      {fp:,} of {tn + fp:,} "
      f"({fp / (tn + fp):.4%})")

## 4. ROC and Precision-Recall curves

Both need *probabilities*, not just labels, so we score the held-out rows
directly with the saved bundle. `_read_rows_by_position` is used deliberately:
it streams the file and returns only the requested rows.

In [ ]:
from sklearn.metrics import (
    roc_curve, auc as compute_roc_auc,
    precision_recall_curve, average_precision_score, roc_auc_score,
)
from src.detection.fraud_model import load_model, engineer_features
from src.evaluation.model_evaluation import _read_rows_by_position

bundle = load_model()
test_rows = _read_rows_by_position(bundle["test_index"])

eng_test = engineer_features(
    test_rows, large_amount_threshold=bundle.get("large_amount_threshold")
)
feat_cols = bundle["feature_cols"]
for c in feat_cols:                       # same defensive fill as predict()
    if c not in eng_test.columns:
        eng_test[c] = 0
X_scaled = bundle["scaler"].transform(eng_test[feat_cols].fillna(0))

y_true = test_rows["isfraud"]
y_score = bundle["model"].predict_proba(X_scaled)[:, 1]

fpr, tpr, _ = roc_curve(y_true, y_score)
pr_precision, pr_recall, _ = precision_recall_curve(y_true, y_score)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(fpr, tpr, lw=2, label="model")
axes[0].plot([0, 1], [0, 1], "k--", lw=1, label="chance")
axes[0].set(xlabel="False positive rate", ylabel="True positive rate (recall)",
            title=f"ROC curve (AUC = {compute_roc_auc(fpr, tpr):.4f})")
axes[0].legend()

axes[1].plot(pr_recall, pr_precision, lw=2, color="tab:orange", label="model")
base_rate = float(y_true.mean())
axes[1].axhline(base_rate, ls="--", c="k", lw=1,
                label=f"baseline = {base_rate:.5f}")
axes[1].set(xlabel="Recall", ylabel="Precision",
            title=f"Precision-Recall (AP = {average_precision_score(y_true, y_score):.4f})")
axes[1].legend()
fig.tight_layout()
plt.show()

## 5. The rule engine on the same rows

A fair, label-free baseline: how much of the job do the explicit rules do alone?
`rules_triggered` (0-4) is used as the ranking score for the ROC-AUC.

In [ ]:
from src.detection.rule_based import apply_fraud_rules

rules = apply_fraud_rules(test_rows)
pred = rules["rule_based_flag"].astype(int)

tp_r = int(((pred == 1) & (y_true == 1)).sum())
fp_r = int(((pred == 1) & (y_true == 0)).sum())
fn_r = int(((pred == 0) & (y_true == 1)).sum())

comparison = pd.DataFrame({
    "ML (Random Forest)": {
        "precision": results["precision"],
        "recall": results["recall"],
        "f1": results["f1"],
        "roc_auc": results["roc_auc"],
    },
    "rule engine": {
        "precision": tp_r / max(1, tp_r + fp_r),
        "recall": tp_r / max(1, tp_r + fn_r),
        "f1": 2 * tp_r / max(1, 2 * tp_r + fp_r + fn_r),
        "roc_auc": roc_auc_score(y_true, rules["rules_triggered"]),
    },
})
comparison.round(4)

## 6. Business view: risk levels

`combine_risk_scores` merges ML probability, rules and anomaly score into one
`risk_score` bucketed as **low / medium / high**. The table below shows, on
held-out rows, how each true class is distributed across the alert buckets -
this is what an operations team would actually see.

In [ ]:
from src.utils.risk_scoring import combine_risk_scores, explain_risk

demo = test_rows.sample(n=1000, random_state=42)
scored = combine_risk_scores(demo.drop(columns=["isfraud"]))
scored.insert(0, "isfraud", demo["isfraud"].values)

print("alert buckets by true class (counts):")
display(pd.crosstab(scored["isfraud"], scored["risk_level"]))
print("row-normalised (%):")
display(pd.crosstab(scored["isfraud"], scored["risk_level"], normalize="index").round(4) * 100)

In [ ]:
# Why was the riskiest row flagged? (always keep the explanation with the score)
worst = scored.sort_values("risk_score", ascending=False).iloc[0]
print("true label:", int(worst["isfraud"]), "| type:", worst["type"],
      "| amount:", f"{worst['amount']:,.2f}")
explain_risk(worst)

## Conclusions

- **Accuracy is misleading** here: predicting "legitimate" always would score
  ~99.9% while catching nothing. Recall/precision/ROC-AUC are the metrics that
  count (see `models/evaluation_results.json` for the current numbers).
- The production model's ROC-AUC (~0.996 in the recorded run) means it *ranks*
  fraud very well; the practical question is where to set the alert threshold,
  trading analyst time (precision) against missed fraud (recall).
- The **rule engine alone is competitive** and infinitely explainable; the ML
  model adds ranking power on cases the rules never anticipate.
- The **risk-level buckets** turn a probability into an operational queue:
  high-risk rows are rare and enriched with fraud, and `explain_risk()` always
  carries the human-readable reasons alongside the score.
- Next steps to consider: threshold tuning on the PR curve, cost-sensitive
  training (class weights / resampling), and periodic retraining as new
  patterns appear.